# Hour 2: State Reducers, Loops & Checkpointing — The Traffic Controller

**Goal:** Master the three concepts that make LangGraph production-ready: state accumulation, graph cycles, and automatic state snapshots.

**What you'll learn:**
- **State Reducers** — fields that *accumulate* instead of overwrite (`Annotated[list, add]`)
- **Conditional Loops** — graph cycles managed by LangGraph (not Python `while` loops)
- **Checkpointing** — automatic state snapshots at every node transition

**Prerequisites:** Complete Hour 1. No API key needed.

**Time:** ~45 minutes

---

### Why This Hour Matters

Hour 1 graphs are linear: START → nodes → END. That's fine for simple routing. But real systems need:
- **Accumulation:** Multiple nodes contributing to the same field without stomping on each other
- **Iteration:** "Keep searching until we have enough data"
- **Audit trails:** "Show me exactly what happened at every step"

Without these, you can't build multi-agent collaboration (Hour 3) or production pipelines (Hour 4).

## The Problem With Overwriting

In Hour 1, every node overwrites state fields. Node A sets `threat_level`, node B sets `action`. Clean and simple — until you need **accumulation.**

Imagine a research pipeline: a retriever finds 3 documents, an analyzer reviews them and adds notes, a second retriever pass finds 2 more. If each node **overwrites** the `documents` field, you lose everything the previous node found.

**The fix is a single annotation:**

```python
from typing import Annotated
from operator import add

# Without reducer — OVERWRITES
query: str

# With reducer — APPENDS
findings: Annotated[list[str], add]
```

The `add` is `operator.add`, which for lists means **concatenation**. When a node returns `{"findings": ["new fact"]}`, LangGraph does:
```python
state["findings"] = state["findings"] + ["new fact"]   # Concatenation, not replacement
```

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ─── STATE WITH REDUCERS ────────────────────────────────────────────────
# Compare to Hour 1's IntelState where every field just overwrites.
#
# Annotated[list[str], add] means:
#   - Type: list[str]
#   - Reducer: operator.add (list concatenation)
#   → Node returns {"findings": ["X"]} → LangGraph does: existing + ["X"]
#
# Fields WITHOUT Annotated (like loop_count) still overwrite normally.

class ResearchState(TypedDict):
    query: str                                    # No reducer → overwrites
    findings: Annotated[list[str], add]           # REDUCER: appends
    sources: Annotated[list[str], add]            # REDUCER: appends
    loop_count: int                                # No reducer → overwrites
    summary: str                                   # No reducer → overwrites

print("ResearchState defined with 2 reducer fields (findings, sources)")
print("and 3 overwrite fields (query, loop_count, summary)")

## The Nodes

Five nodes form the pipeline. Notice how `search_web` and `analyze_findings` both write to `findings` — because of the reducer, their contributions **accumulate** instead of overwriting each other.

```
search_web → returns {"findings": ["fact1", "fact2"]}     → state: ["fact1", "fact2"]
analyze_findings → returns {"findings": ["analysis"]}     → state: ["fact1", "fact2", "analysis"]
search_web (loop 2) → returns {"findings": ["fact3"]}     → state: ["fact1", "fact2", "analysis", "fact3"]
```

Nothing is lost. That's the power of reducers.

In [ ]:
def search_web(state: ResearchState) -> dict:
    """
    NODE: search_web
    Reads: query, loop_count
    Writes: findings (APPENDS via reducer), sources (APPENDS via reducer)

    In production, replace with: results = tavily_client.search(query)
    """
    query = state["query"]
    loop = state.get("loop_count", 0)
    print(f"  [search_web] Loop {loop}: Searching for '{query}'")

    # Simulated results — each loop "discovers" different information
    if loop == 0:
        return {
            "findings": [
                "LangGraph is a library for building stateful multi-agent apps",
                "It uses directed graphs where nodes are functions",
            ],
            "sources": ["langgraph-docs", "blog-post-1"],
        }
    else:
        return {
            "findings": [
                "LangGraph supports checkpointing for pause/resume workflows",
                "State reducers allow accumulation across nodes",
            ],
            "sources": ["langgraph-advanced-docs", "github-examples"],
        }


def analyze_findings(state: ResearchState) -> dict:
    """
    NODE: analyze_findings
    Reads: findings (the FULL accumulated list so far)
    Writes: findings (APPENDS an analysis note via reducer)

    IMPORTANT: Returns a LIST, not a bare string.
    The reducer does list + list. Returning a string causes TypeError.
    """
    num_findings = len(state["findings"])
    print(f"  [analyze] Reviewing {num_findings} findings so far")
    return {
        "findings": [f"Analysis: {num_findings} facts gathered, assessing completeness"],
    }


def quality_gate(state: ResearchState) -> str:
    """
    ROUTER: Decides whether to loop or finish.

    This is the conditional edge that enables LOOPING.
    One return value points BACKWARD in the graph → creates a cycle.

    Guard conditions prevent infinite loops:
    - >= 5 findings → enough data, finish
    - >= 2 loops → safety limit reached, finish anyway
    """
    num_findings = len(state["findings"])
    loop_count = state.get("loop_count", 0)
    print(f"  [quality_gate] Findings: {num_findings}, Loops: {loop_count}")

    if num_findings >= 5 or loop_count >= 2:
        return "summarize"
    return "loop_back"


def increment_loop(state: ResearchState) -> dict:
    """
    NODE: increment_loop — Utility node that bumps the loop counter.

    Why separate from search_web? Separation of concerns.
    Search does searching. Loop management is a different job.
    """
    current = state.get("loop_count", 0)
    print(f"  [increment_loop] {current} → {current + 1}")
    return {"loop_count": current + 1}


def summarize(state: ResearchState) -> dict:
    """
    NODE: summarize — Produces final output from ALL accumulated findings.

    This is the payoff: because findings used a reducer,
    we have the COMPLETE picture here — every fact from every loop.
    """
    all_findings = state["findings"]
    all_sources = state["sources"]
    summary = (
        f"Research complete.\n"
        f"  Total findings: {len(all_findings)}\n"
        f"  Sources consulted: {', '.join(all_sources)}\n"
        f"  Key facts:\n"
    )
    for i, finding in enumerate(all_findings, 1):
        summary += f"    {i}. {finding}\n"
    return {"summary": summary}

print("All 5 nodes defined: search_web, analyze_findings, quality_gate, increment_loop, summarize")

## Wiring the Graph — With a Loop

The key difference from Hour 1: one edge points **backward**.

```
increment_loop → search_web    ← This creates a CYCLE
```

This is NOT a Python `while` loop. It's a graph cycle managed by LangGraph. The difference matters:
- LangGraph **checkpoints** state at every node boundary during the loop
- The loop can be **interrupted and resumed** (human-in-the-loop)
- You can **inspect** the state at any iteration after the fact
- A Python `while` loop offers none of these

> **Always include a loop guard.** Without one, a conditional loop can run forever. Use a retry counter and have your router check it.

In [ ]:
# ─── WIRE THE GRAPH ─────────────────────────────────────────────────────
graph = StateGraph(ResearchState)

# Register all nodes
graph.add_node("search_web", search_web)
graph.add_node("analyze_findings", analyze_findings)
graph.add_node("increment_loop", increment_loop)
graph.add_node("summarize", summarize)

# Fixed edges
graph.add_edge(START, "search_web")
graph.add_edge("search_web", "analyze_findings")
graph.add_edge("increment_loop", "search_web")      # ← THE LOOP
graph.add_edge("summarize", END)

# Conditional edge: quality gate decides loop or finish
graph.add_conditional_edges(
    "analyze_findings",
    quality_gate,
    {
        "summarize": "summarize",
        "loop_back": "increment_loop",
    },
)

# ─── COMPILE WITH CHECKPOINTING ─────────────────────────────────────────
# MemorySaver = in-memory snapshots (dev/testing)
# SqliteSaver = local file (small apps)
# PostgresSaver = production (multi-user, persistent)
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("Graph compiled with MemorySaver checkpointing!")

## Running the Pipeline

Watch the loop in action. The quality gate will evaluate after each analysis pass, and either loop back for more research or proceed to summarize.

**What to watch for:**
- Findings **accumulate** across loops (reducer in action)
- The loop counter increments each pass (overwrite field)
- The quality gate exits when it has enough data

In [ ]:
print("=" * 60)
print("RUNNING RESEARCH PIPELINE")
print("=" * 60)

# Config with thread_id — REQUIRED when using a checkpointer
# Same thread = same state history. Different thread = fresh start.
config = {"configurable": {"thread_id": "research-session-1"}}

result = app.invoke(
    {
        "query": "What is LangGraph and why use it?",
        "findings": [],       # Start empty — reducer will accumulate
        "sources": [],        # Start empty — reducer will accumulate
        "loop_count": 0,
        "summary": "",
    },
    config=config,
)

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(result["summary"])

## Inspecting the Checkpoint (Your Audit Trail)

Because we compiled with a checkpointer, LangGraph saved a snapshot at **every node transition**. This is your audit log — you can see exactly what the state looked like at any point.

In an AI Security context, this is critical:
- **Audit trail:** Inspect what happened at each step
- **Replay:** Re-run from any checkpoint with different inputs
- **Human-in-the-loop:** Pause at a gate node, wait for approval, resume

In [ ]:
print("=" * 60)
print("CHECKPOINT INSPECTION")
print("=" * 60)
state_snapshot = app.get_state(config)
print(f"  Total findings accumulated: {len(state_snapshot.values['findings'])}")
print(f"  Sources: {state_snapshot.values['sources']}")
print(f"  Loop count: {state_snapshot.values['loop_count']}")
print(f"\n  All findings:")
for i, f in enumerate(state_snapshot.values['findings'], 1):
    print(f"    {i}. {f}")

## Key Takeaways

| Concept | What It Does | Why It Matters |
|---|---|---|
| **State Reducers** | `Annotated[list, add]` makes fields accumulate | Multi-agent collaboration without data loss |
| **Conditional Loops** | Router returns backward edge → graph cycle | LangGraph manages iteration with checkpoints |
| **Loop Guards** | Retry counter in state, checked by router | Prevents infinite loops |
| **Checkpointing** | `MemorySaver()` → automatic state snapshots | Audit trail, replay, human-in-the-loop |
| **Thread IDs** | `{"thread_id": "..."}` in config | Session management (same thread = same history) |

**Common mistake to avoid:** Returning a bare string when the reducer expects a list. `operator.add` does list concatenation, so always return `["item"]` not `"item"`.

**What's next:** In Hour 3, we replace the simulated search with actual LLM calls and build the supervisor pattern — one AI routing tasks to specialist AIs.

---
*Next: [03_multi_agent.ipynb](./03_multi_agent.ipynb) — LLMs inside nodes and the supervisor pattern*